# 02 — Results Comparison

**Progetto:** Super-Resolution su LSUN Church  
**Obiettivo:** Confronto quantitativo (PSNR, SSIM) e qualitativo (visualizzazione) dei tre metodi implementati:
- **Metodo 1:** Total Variation (Variazionale)
- **Metodo 2:** U-Net End-to-End
- **Metodo 3:** DiffPIR (Generativo)

> ⚠️ **Nota:** Questo notebook richiede che i metodi di `src/methods/` siano implementati e che il test set degradato sia disponibile in `data/`.

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
import torch
import pandas as pd

from normalization import default_normalizer
from degradations import DegradationOperator

# Importa metriche (da implementare in src/metrics.py)
try:
    from metrics import compute_psnr, compute_ssim
    METRICS_AVAILABLE = True
    print('Metriche disponibili: OK')
except ImportError:
    METRICS_AVAILABLE = False
    print('⚠️  src/metrics.py non ancora implementato — le metriche non saranno disponibili')

# Importa metodi (da implementare)
METHODS_AVAILABLE = {}
try:
    from methods.tv import tv_reconstruction
    METHODS_AVAILABLE['TV'] = tv_reconstruction
    print('Metodo TV: OK')
except ImportError:
    print('⚠️  src/methods/tv.py non ancora implementato')

try:
    from methods.unet import UNet
    METHODS_AVAILABLE['UNet'] = UNet
    print('Metodo UNet: OK')
except ImportError:
    print('⚠️  src/methods/unet.py non ancora implementato')

try:
    from methods.diffpir import diffpir_reconstruction
    METHODS_AVAILABLE['DiffPIR'] = diffpir_reconstruction
    print('Metodo DiffPIR: OK')
except ImportError:
    print('⚠️  src/methods/diffpir.py non ancora implementato')

print(f'\nMetodi disponibili: {list(METHODS_AVAILABLE.keys()) or "Nessuno (da implementare)"}')
print(f'PyTorch version: {torch.__version__}')

## 1. Configurazione esperimenti

In [ ]:
data_dir = Path('..') / 'data'

# Configurazioni di degradazione testate
CONFIGS = [
    {'sr_factor': 2, 'noise_std': 0.005, 'name': 'SR2x_n0005', 'label': 'SR 2x, σ=0.005'},
    {'sr_factor': 2, 'noise_std': 0.010, 'name': 'SR2x_n001',  'label': 'SR 2x, σ=0.010'},
    {'sr_factor': 4, 'noise_std': 0.005, 'name': 'SR4x_n0005', 'label': 'SR 4x, σ=0.005'},
    {'sr_factor': 4, 'noise_std': 0.010, 'name': 'SR4x_n001',  'label': 'SR 4x, σ=0.010'},
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Controlla disponibilità test set
test_dir = data_dir / 'test'
if test_dir.exists():
    test_files = sorted(test_dir.glob('*.png'))
    print(f'Test set: {len(test_files)} immagini in {test_dir}')
else:
    print('⚠️  Test set non trovato. Esegui: python src/dataset.py')

## 2. Baseline — Input Bicubico (Upsampling semplice)

Confronto baseline: PSNR/SSIM dell'immagine degradata upsampled con interpolazione bicubica (senza ricostruzione).

In [ ]:
# Placeholder — verrà popolato con dati reali una volta implementate le metriche

if METRICS_AVAILABLE and test_dir.exists():
    test_files_sample = sorted(test_dir.glob('*.png'))[:50]  # usa 50 immagini per velocità
    
    baseline_results = {cfg['name']: {'psnr': [], 'ssim': []} for cfg in CONFIGS}
    
    for img_path in test_files_sample:
        hr_img = Image.open(img_path).convert('RGB')
        hr_np = np.array(hr_img).astype(np.float32) / 255.0
        
        for cfg in CONFIGS:
            op = DegradationOperator(sr_factor=cfg['sr_factor'], noise_std=cfg['noise_std'], seed=42)
            deg_tensor = op(hr_img)  # (C, H_lr, W_lr)
            
            # Upsample bicubico
            deg_img = Image.fromarray(
                (deg_tensor.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)
            )
            up_img = deg_img.resize((256, 256), Image.Resampling.BICUBIC)
            up_np = np.array(up_img).astype(np.float32) / 255.0
            
            psnr_val = compute_psnr(hr_np, up_np)
            ssim_val = compute_ssim(hr_np, up_np)
            baseline_results[cfg['name']]['psnr'].append(psnr_val)
            baseline_results[cfg['name']]['ssim'].append(ssim_val)
    
    print('=== Baseline Bicubico ===')
    print(f'{"Config":20s} {"PSNR (dB)":>10s} {"SSIM":>10s}')
    print('-' * 42)
    for cfg in CONFIGS:
        psnr = np.mean(baseline_results[cfg['name']]['psnr'])
        ssim = np.mean(baseline_results[cfg['name']]['ssim'])
        print(f"{cfg['label']:20s} {psnr:10.2f} {ssim:10.4f}")
else:
    print('ℹ️  Baseline verrà calcolata dopo implementazione di src/metrics.py')
    print()
    print('Tabella risultati attesa (da riempire):')
    print(f'{"Config":20s} {"PSNR Baseline":>14s} {"PSNR TV":>10s} {"PSNR UNet":>10s} {"PSNR DiffPIR":>12s}')
    print('-' * 68)
    for cfg in CONFIGS:
        print(f"{cfg['label']:20s} {'---':>14s} {'---':>10s} {'---':>10s} {'---':>12s}")

## 3. Confronto Qualitativo — Visualizzazione

In [ ]:
# Sceglie un'immagine di test rappresentativa
if test_dir.exists() and sorted(test_dir.glob('*.png')):
    sample_path = sorted(test_dir.glob('*.png'))[0]
    hr_img = Image.open(sample_path).convert('RGB')
else:
    # Fallback sintetico
    arr = np.random.randint(80, 200, (256, 256, 3), dtype=np.uint8)
    for i in range(0, 256, 32):
        arr[i:i+8, :] = [160, 100, 50]
    hr_img = Image.fromarray(arr)
    sample_path = Path('synthetic')

print(f'Immagine campione: {sample_path.name}')

# Usa la config 4x + noise 0.01 (caso più difficile)
cfg_hard = CONFIGS[3]  # SR 4x, σ=0.01
op = DegradationOperator(sr_factor=cfg_hard['sr_factor'], noise_std=cfg_hard['noise_std'], seed=42)
deg_tensor = op(hr_img)
deg_np = (deg_tensor.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)
deg_img = Image.fromarray(deg_np)

# Bicubic upsampling
bicubic_up = deg_img.resize((256, 256), Image.Resampling.BICUBIC)

# Costruisci lista risultati (metodi da implementare usano placeholder)
results = {
    'Ground Truth': np.array(hr_img),
    f'Degradato\n({cfg_hard["label"]})': deg_np,
    'Bicubic\n(baseline)': np.array(bicubic_up),
}

# Aggiunge risultati metodi se disponibili
if 'TV' in METHODS_AVAILABLE:
    tv_result = METHODS_AVAILABLE['TV'](deg_tensor, sr_factor=cfg_hard['sr_factor'])
    results['TV'] = (tv_result.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)

# Plot
n_cols = len(results)
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
if n_cols == 1:
    axes = [axes]

for ax, (title, img_arr) in zip(axes, results.items()):
    h, w = img_arr.shape[:2]
    ax.imshow(img_arr)
    ax.set_title(f'{title}\n{w}×{h}', fontsize=10)
    ax.axis('off')

plt.suptitle(f'Confronto Metodi — Configurazione: {cfg_hard["label"]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot salvato in data/results_comparison.png')

## 4. Tabella Riepilogativa Risultati

Questa tabella verrà popolata automaticamente una volta implementati tutti i metodi.

In [ ]:
# Template tabella risultati — da riempire con valori reali
results_data = {
    'Configurazione': [cfg['label'] for cfg in CONFIGS],
    'PSNR Bicubic (dB)': ['—'] * 4,
    'PSNR TV (dB)': ['—'] * 4,
    'PSNR UNet (dB)': ['—'] * 4,
    'PSNR DiffPIR (dB)': ['—'] * 4,
    'SSIM Bicubic': ['—'] * 4,
    'SSIM TV': ['—'] * 4,
    'SSIM UNet': ['—'] * 4,
    'SSIM DiffPIR': ['—'] * 4,
}

df = pd.DataFrame(results_data)
print('=== Tabella Risultati (PSNR e SSIM) ===')
print(df.to_string(index=False))
print('\nℹ️  I valori verranno aggiornati man mano che i metodi vengono implementati.')

---
## Checklist Implementazione

| Componente | File | Stato |
|:---|:---|:---:|
| Metriche PSNR/SSIM | `src/metrics.py` | ❌ Da fare |
| Total Variation | `src/methods/tv.py` | ❌ Da fare |
| U-Net | `src/methods/unet.py` | ❌ Da fare |
| Training U-Net | `src/train_unet.py` | ❌ Da fare |
| DiffPIR | `src/methods/diffpir.py` | ❌ Da fare |